# segment-line-intersect-2d — worked example 2: Detect a segment that ends before the line (no hit)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `segment-line-intersect-2d`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

When the parametric parameter t_seg is outside [0, 1], it means the line's intersection point falls on the infinite extension of the segment, not the segment itself. For example if t_seg = 1.5, the intersection is 50% past the endpoint S1. The same 2×2 solve still works — only the hit test changes the boolean outcome.

## Worked solution

**Step 1 — Set up a geometry where the intersection would be past S1.** A short segment from (0,0) to (1,0) and a vertical line at x=2. The extension of the segment hits x=2 at t=2.0, but the actual segment only goes to t=1.0.

**Step 2 — Solve the system exactly as before.** The solve itself doesn't know or care whether the intersection is on the segment.

**Step 3 — Check t_seg.** We get t_seg=2.0, which fails the [0,1] test, so `hit = False`.

**Step 4 — Confirm with a segment that does reach.** Extend S1 to (3,0) — now t_seg = 2/3, which is in [0,1], so `hit = True`. We run both cases and compare.

In [ ]:
import torch as t

t.manual_seed(0)

def seg_line_intersect(S0, S1, L0, L1):
    d = S1 - S0
    e = L1 - L0
    A = t.stack([d, -e], dim=1)
    b = L0 - S0
    ts = t.linalg.solve(A, b)
    t_seg = ts[0].item()
    hit = (0.0 <= t_seg <= 1.0)
    return t_seg, hit

L0 = t.tensor([2.0, -1.0])
L1 = t.tensor([2.0,  3.0])  # vertical line at x=2

# Short segment -- doesn't reach x=2
S0 = t.tensor([0.0, 0.0])
S1_short = t.tensor([1.0, 0.0])
t1, hit1 = seg_line_intersect(S0, S1_short, L0, L1)
print(f'Short segment: t_seg={t1:.3f}, hit={hit1}')  # t_seg=2.0, hit=False

# Long segment -- does reach x=2
S1_long = t.tensor([3.0, 0.0])
t2, hit2 = seg_line_intersect(S0, S1_long, L0, L1)
print(f'Long segment:  t_seg={t2:.3f}, hit={hit2}')  # t_seg~0.667, hit=True